In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time
import random
from urllib.parse import urljoin, urlparse
from pathlib import Path
import logging
import hashlib
from PIL import Image
import json

class ImprovedLamodaScraper:
    def __init__(self, download_path="./images"):
        self.download_path = Path(download_path)
        self.setup_logging()
        self.session = requests.Session()
        self.downloaded_urls = set()  # Track downloaded image URLs
        self.downloaded_hashes = set()  # Track image content hashes
        self.product_ids = set()  # Track product IDs to avoid duplicates
        self.stats = {
            'total_processed': 0,
            'duplicates_skipped': 0,
            'download_errors': 0,
            'unique_downloaded': 0,
            'underwear_skipped': 0
        }

    def setup_logging(self):
        """Setup logging configuration"""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('improved_lamoda_scraper.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def get_user_agent(self):
        """Return a random user agent string to avoid detection."""
        user_agents = [
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15',
            'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:89.0) Gecko/20100101 Firefox/89.0'
        ]
        return random.choice(user_agents)

    def get_image_hash(self, image_content):
        """Generate hash for image content to detect duplicates."""
        return hashlib.md5(image_content).hexdigest()

    def get_high_res_image_url(self, image_url):
        """Convert low-res image URL to high-res version."""
        # Lamoda typically serves different sizes, try to get the largest
        if 'lamoda' in image_url:
            # Replace size parameters to get larger images
            image_url = image_url.replace('_120x120', '_600x866')
            image_url = image_url.replace('_240x240', '_600x866')
            image_url = image_url.replace('_300x300', '_600x866')
            image_url = image_url.replace('_150x150', '_600x866')
        return image_url

    def extract_product_id(self, product_url_or_element):
        """Extract unique product ID from URL or element."""
        try:
            # Try to find product ID in various ways
            if isinstance(product_url_or_element, str):
                # Extract from URL
                if '/p/' in product_url_or_element:
                    return product_url_or_element.split('/p/')[1].split('/')[0]
            else:
                # Try to find data attributes or links
                link_elem = product_url_or_element.find('a', href=True)
                if link_elem and '/p/' in link_elem['href']:
                    return link_elem['href'].split('/p/')[1].split('/')[0]
                
                # Look for data attributes
                for attr in ['data-product-id', 'data-id', 'id']:
                    if product_url_or_element.get(attr):
                        return product_url_or_element[attr]
            
            return None
        except:
            return None

    def is_valid_image_url(self, url):
        """Check if URL is a valid image URL."""
        if not url:
            return False
        
        # Skip placeholder, loading, or very small images
        skip_keywords = ['placeholder', 'loading', 'blank', 'spacer', '1x1', 'pixel']
        url_lower = url.lower()
        
        for keyword in skip_keywords:
            if keyword in url_lower:
                return False
        
        # Must be a reasonable image format
        valid_extensions = ['.jpg', '.jpeg', '.png', '.webp']
        return any(ext in url_lower for ext in valid_extensions)

    def detect_pagination_type(self, soup):
        """Detect if site uses traditional pagination or infinite scroll."""
        # Look for pagination indicators
        pagination_selectors = [
            '.pagination',
            '[class*="pagination"]',
            '.pager',
            '[class*="pager"]',
            'a[href*="page="]',
            'button[data-page]'
        ]
        
        for selector in pagination_selectors:
            if soup.select(selector):
                return "traditional"
        
        # Look for infinite scroll indicators
        infinite_scroll_indicators = [
            'data-infinite-scroll',
            'class*="infinite"',
            'id*="infinite"',
            'data-lazy-load',
            '[class*="load-more"]'
        ]
        
        for indicator in infinite_scroll_indicators:
            if soup.select(f'[{indicator}]'):
                return "infinite"
        
        return "unknown"

    def get_max_page_number(self, soup):
        """Try to determine the maximum number of pages available."""
        # Look for pagination links
        page_links = soup.find_all('a', href=lambda x: x and 'page=' in x)
        max_page = 1
        
        for link in page_links:
            try:
                href = link.get('href', '')
                if 'page=' in href:
                    page_num = int(href.split('page=')[1].split('&')[0])
                    max_page = max(max_page, page_num)
            except:
                continue
        
        # Also check for numbered pagination
        page_numbers = soup.find_all(text=lambda text: text and text.isdigit())
        for num_text in page_numbers[-10:]:  # Check last 10 numbers found
            try:
                num = int(num_text)
                if 2 <= num <= 1000:  # Reasonable page number range
                    max_page = max(max_page, num)
            except:
                continue
        
        return min(max_page, 100)  # Cap at reasonable limit

    def is_underwear_product(self, product_element, product_name=""):
        """Check if product is underwear and should be skipped."""
        underwear_keywords = [
            # English terms
            'underwear', 'lingerie', 'panties', 'bra', 'thong', 'boxer', 'briefs', 
            'bikini', 'swimsuit', 'swimwear', 'undergarment', 'intimate', 'sleepwear',
            'nightwear', 'pyjama', 'pajama', 'nightgown', 'nightdress', 'slip',
            'camisole', 'undershirt', 'tank top', 'vest', 'corset', 'bodysuit',
            'shapewear', 'hosiery', 'stockings', 'tights', 'pantyhose', 'socks',
            
            # Russian/Kazakh terms (common on Lamoda.kz)
            'белье', 'трусы', 'лифчик', 'бюстгальтер', 'стринги', 'боксеры',
            'плавки', 'купальник', 'нижнее', 'интимн', 'пижам', 'ночн',
            'майк', 'футболк', 'корсет', 'колготки', 'носки', 'чулки',
            'боди', 'комбинаци', 'сорочк', 'халат', 'домашн',
            
            # Category indicators
            'sleepwear', 'intimates', 'lounge', 'home wear', 'homewear'
        ]
        
        # Check product name/title
        product_text = product_name.lower()
        
        # Also check any text content in the product element
        if product_element:
            try:
                element_text = product_element.get_text().lower()
                product_text += " " + element_text
            except:
                pass
        
        # Check for underwear keywords
        for keyword in underwear_keywords:
            if keyword in product_text:
                return True
        
        return False

    def download_image(self, url, category_folder, filename, product_id=None):
        """Download an image from URL and save it to disk."""
        try:
            self.stats['total_processed'] += 1
            
            # Check if URL already processed
            if url in self.downloaded_urls:
                self.logger.info(f"URL already processed: {url}")
                self.stats['duplicates_skipped'] += 1
                return None

            # Check if product ID already processed
            if product_id and product_id in self.product_ids:
                self.logger.info(f"Product ID already processed: {product_id}")
                self.stats['duplicates_skipped'] += 1
                return None

            # Create category folder
            category_path = self.download_path / category_folder
            category_path.mkdir(parents=True, exist_ok=True)

            # Fix for protocol-relative URLs
            if url.startswith('//'):
                url = 'https:' + url

            # Try to get high-res version
            url = self.get_high_res_image_url(url)

            headers = {
                'User-Agent': self.get_user_agent(),
                'Referer': 'https://www.lamoda.kz/',
                'Accept': 'image/webp,image/apng,image/*,*/*;q=0.8'
            }

            response = self.session.get(url, headers=headers, timeout=30)
            if response.status_code == 200 and len(response.content) > 1000:  # Skip tiny images
                
                # Check content hash for duplicates
                content_hash = self.get_image_hash(response.content)
                if content_hash in self.downloaded_hashes:
                    self.logger.info(f"Duplicate image content detected: {filename}")
                    self.stats['duplicates_skipped'] += 1
                    return None

                # Clean filename
                clean_filename = "".join(c for c in filename if c.isalnum() or c in (' ', '-', '_', '.')).rstrip()
                if not any(clean_filename.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.webp']):
                    clean_filename += '.jpg'

                # Add product ID to filename if available
                if product_id:
                    name_part = clean_filename.rsplit('.', 1)[0]
                    ext_part = clean_filename.rsplit('.', 1)[1] if '.' in clean_filename else 'jpg'
                    clean_filename = f"{name_part}_{product_id}.{ext_part}"

                file_path = category_path / clean_filename

                # Skip if file already exists and has reasonable size
                if file_path.exists() and file_path.stat().st_size > 1000:
                    self.logger.info(f"Image already exists: {clean_filename}")
                    return str(file_path)

                # Save image
                with open(file_path, 'wb') as f:
                    f.write(response.content)

                # Try to verify it's a valid image
                try:
                    with Image.open(file_path) as img:
                        width, height = img.size
                        if width < 50 or height < 50:  # Skip very small images
                            os.remove(file_path)
                            self.logger.info(f"Removed too small image: {clean_filename} ({width}x{height})")
                            return None
                except:
                    # If PIL can't open it, it's probably not a valid image
                    if file_path.exists():
                        os.remove(file_path)
                    self.logger.warning(f"Invalid image file removed: {clean_filename}")
                    return None

                # Track successful download
                self.downloaded_urls.add(url)
                self.downloaded_hashes.add(content_hash)
                if product_id:
                    self.product_ids.add(product_id)
                
                self.stats['unique_downloaded'] += 1
                self.logger.info(f"Downloaded: {clean_filename} ({width}x{height})")
                return str(file_path)
            else:
                self.logger.warning(f"Failed to download or image too small: {url}, Status: {response.status_code}, Size: {len(response.content) if response.status_code == 200 else 'N/A'}")
                self.stats['download_errors'] += 1
                return None
                
        except Exception as e:
            self.logger.error(f"Error downloading image {url}: {e}")
            self.stats['download_errors'] += 1
            return None

    def scrape_url(self, url, category_name, max_items=100, max_pages=10):
        """Scrape product images from Lamoda URL."""
        downloaded_count = 0
        processed_products = set()
        base_url = url  # Store original URL
        consecutive_empty_pages = 0
        pagination_type = "unknown"
        detected_max_pages = max_pages

        self.logger.info(f"Starting scraping: {url} (max: {max_items} items, {max_pages} pages)")

        # First, do a test request to detect pagination type
        try:
            test_headers = {
                'User-Agent': self.get_user_agent(),
                'Accept-Language': 'en-US,en;q=0.9',
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                'Referer': 'https://www.lamoda.kz/'
            }
            test_response = self.session.get(base_url, headers=test_headers, timeout=30)
            if test_response.status_code == 200:
                test_soup = BeautifulSoup(test_response.text, "html.parser")
                pagination_type = self.detect_pagination_type(test_soup)
                detected_max_pages = min(self.get_max_page_number(test_soup), max_pages)
                self.logger.info(f"Detected pagination type: {pagination_type}, max pages: {detected_max_pages}")
        except Exception as e:
            self.logger.warning(f"Could not detect pagination: {e}")

        for page in range(1, detected_max_pages + 1):
            if downloaded_count >= max_items:
                break

            # Always use the original base URL for pagination
            if '?' in base_url:
                page_url = f"{base_url}&page={page}"
            else:
                page_url = f"{base_url}?page={page}"
            
            # Validate URL before making request
            if not page_url.startswith('http'):
                self.logger.error(f"Invalid URL format: {page_url}")
                break
                
            self.logger.info(f"Scraping page {page}: {page_url}")

            try:
                # Random delay between requests
                time.sleep(random.uniform(2, 4))

                headers = {
                    'User-Agent': self.get_user_agent(),
                    'Accept-Language': 'en-US,en;q=0.9',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                    'Referer': 'https://www.lamoda.kz/'
                }

                response = self.session.get(page_url, headers=headers, timeout=30)
                if response.status_code != 200:
                    self.logger.warning(f"Failed to retrieve page {page}: Status code {response.status_code}")
                    consecutive_empty_pages += 1
                    if consecutive_empty_pages >= 3:
                        self.logger.info("Too many failed requests, stopping pagination")
                        break
                    continue

                soup = BeautifulSoup(response.text, "html.parser")

                # Check if this is actually a valid Lamoda page
                if not any(domain in response.url.lower() for domain in ['lamoda.kz', 'lamoda.ru', 'lmcdn.ru']):
                    self.logger.error(f"Redirected to invalid domain: {response.url}")
                    break

                # Check if we're getting the same content (infinite scroll exhausted)
                if page > 1:
                    # Simple check: if page title/content looks identical, we might be done
                    page_title = soup.find('title')
                    if page_title and 'page not found' in page_title.text.lower():
                        self.logger.info("Reached end of available pages")
                        break

                # Find product containers with multiple selectors
                selectors_to_try = [
                    "div[class*='x-product-card']",
                    "article[class*='product']",
                    "div[class*='product-card']",
                    "[data-product-id]",
                    "a[href*='/p/']"
                ]

                product_cards = []
                for selector in selectors_to_try:
                    product_cards = soup.select(selector)
                    if product_cards:
                        self.logger.info(f"Found {len(product_cards)} products using selector: {selector}")
                        break

                if not product_cards:
                    # Try to find any links to product pages
                    product_links = soup.find_all('a', href=lambda x: x and '/p/' in x)
                    if product_links:
                        self.logger.info(f"Found {len(product_links)} product links as fallback")
                        product_cards = [link.parent for link in product_links if link.parent]

                if not product_cards:
                    self.logger.warning(f"No product cards found on page {page}")
                    consecutive_empty_pages += 1
                    # If no products found on consecutive pages, likely reached end
                    if consecutive_empty_pages >= 2:
                        self.logger.info("No products found on consecutive pages, stopping")
                        break
                    continue
                else:
                    consecutive_empty_pages = 0  # Reset counter on successful page

                page_downloaded = 0
                for i, card in enumerate(product_cards):
                    if downloaded_count >= max_items:
                        break

                    try:
                        # Extract product ID for duplicate checking
                        product_id = self.extract_product_id(card)
                        if product_id in processed_products:
                            continue
                        
                        # Find image element
                        img_selectors = [
                            "img[class*='product']",
                            "img[src*='lamoda']",
                            "img[data-src*='lamoda']",
                            "img"
                        ]

                        img_elem = None
                        image_url = None
                        
                        for img_selector in img_selectors:
                            img_elements = card.select(img_selector)
                            for img in img_elements:
                                # Check src and data-src
                                for src_attr in ['src', 'data-src', 'data-original']:
                                    url = img.get(src_attr)
                                    if self.is_valid_image_url(url):
                                        img_elem = img
                                        image_url = url
                                        break
                                if image_url:
                                    break
                            if image_url:
                                break

                        if not image_url:
                            continue

                        if image_url.startswith('//'):
                            image_url = 'https:' + image_url

                        # Get product name for filename and underwear checking
                        name_selectors = [
                            "[class*='product-name']",
                            "[class*='brand-name']",
                            "[class*='title']",
                            "h1", "h2", "h3"
                        ]

                        product_name = f"product_{downloaded_count + 1}"
                        for name_selector in name_selectors:
                            name_elem = card.select_one(name_selector)
                            if name_elem and name_elem.text.strip():
                                product_name = name_elem.text.strip()[:50]  # Limit length
                                break

                        # Skip underwear products
                        if self.is_underwear_product(card, product_name):
                            self.logger.info(f"Skipping underwear product: {product_name}")
                            self.stats['underwear_skipped'] += 1
                            continue

                        # Create unique filename
                        filename = f"{product_name}_{downloaded_count + 1}.jpg"

                        # Download image
                        if self.download_image(image_url, category_name, filename, product_id):
                            downloaded_count += 1
                            page_downloaded += 1
                            if product_id:
                                processed_products.add(product_id)

                        # Small delay between downloads
                        time.sleep(random.uniform(0.5, 1.5))

                    except Exception as e:
                        self.logger.warning(f"Error processing product {i+1} on page {page}: {e}")
                        continue

                self.logger.info(f"Page {page}: Downloaded {page_downloaded} new images (Total: {downloaded_count}/{max_items})")

                # If no products downloaded from this page, increment counter
                if page_downloaded == 0:
                    consecutive_empty_pages += 1
                    if consecutive_empty_pages >= 2:
                        self.logger.info("No new images downloaded from consecutive pages, stopping")
                        break
                # For infinite scroll sites, try alternative pagination
                if pagination_type == "infinite" and page_downloaded == 0 and page > 3:
                    self.logger.info("Infinite scroll site may be exhausted, trying offset-based pagination")
                    # Try different pagination formats
                    alternative_urls = [
                        f"{base_url}&offset={page * 60}",  # Common offset approach
                        f"{base_url}&start={page * 60}",
                        f"{base_url}&from={(page-1) * 60}"
                    ]
                    
                    for alt_url in alternative_urls:
                        try:
                            alt_response = self.session.get(alt_url, headers=headers, timeout=30)
                            if alt_response.status_code == 200:
                                alt_soup = BeautifulSoup(alt_response.text, "html.parser")
                                alt_cards = alt_soup.select("div[class*='x-product-card']")
                                if len(alt_cards) > len(product_cards):
                                    self.logger.info(f"Found more products with alternative URL: {alt_url}")
                                    soup = alt_soup
                                    product_cards = alt_cards
                                    break
                        except:
                            continue
                else:
                    consecutive_empty_pages = 0  # Reset counter on successful downloads

            except Exception as e:
                self.logger.error(f"Error scraping page {page}: {e}")
                continue

        self.logger.info(f"Finished scraping {category_name}: {downloaded_count} unique images downloaded")
        return downloaded_count

    def scrape_urls(self, urls, max_items_per_url=100, max_pages_per_url=10, base_dir=None):
        """Scrape multiple URLs."""
        if base_dir:
            self.download_path = Path(base_dir)

        results = {}
        
        # Load existing state if available
        state_file = self.download_path / 'scraper_state.json'
        if state_file.exists():
            try:
                with open(state_file, 'r') as f:
                    state = json.load(f)
                    self.downloaded_urls = set(state.get('downloaded_urls', []))
                    self.downloaded_hashes = set(state.get('downloaded_hashes', []))
                    self.product_ids = set(state.get('product_ids', []))
                    self.logger.info(f"Loaded state: {len(self.downloaded_urls)} URLs, {len(self.downloaded_hashes)} hashes, {len(self.product_ids)} products")
            except:
                self.logger.warning("Could not load previous state")

        for i, url in enumerate(urls):
            # Generate category name
            category_name = f"category_{i+1}"
            try:
                if '/c/' in url:
                    url_parts = url.split('/c/')[1].split('/')
                    if len(url_parts) > 1:
                        category_name = url_parts[1].replace('-', '_')
            except:
                pass

            self.logger.info(f"Starting URL {i+1}/{len(urls)}: {category_name}")

            count = self.scrape_url(url, category_name, max_items_per_url, max_pages_per_url)
            results[url] = {
                'category_name': category_name,
                'downloaded_count': count
            }
            
            # Save state after each URL
            try:
                state = {
                    'downloaded_urls': list(self.downloaded_urls),
                    'downloaded_hashes': list(self.downloaded_hashes),
                    'product_ids': list(self.product_ids)
                }
                os.makedirs(self.download_path, exist_ok=True)
                with open(state_file, 'w') as f:
                    json.dump(state, f)
            except:
                self.logger.warning("Could not save state")

            # Delay between URLs
            if i < len(urls) - 1:
                time.sleep(random.uniform(10, 15))

        # Print final statistics
        self.logger.info("="*50)
        self.logger.info("FINAL STATISTICS")
        self.logger.info("="*50)
        self.logger.info(f"Total processed: {self.stats['total_processed']}")
        self.logger.info(f"Unique downloaded: {self.stats['unique_downloaded']}")
        self.logger.info(f"Duplicates skipped: {self.stats['duplicates_skipped']}")
        self.logger.info(f"Underwear skipped: {self.stats['underwear_skipped']}")
        self.logger.info(f"Download errors: {self.stats['download_errors']}")
        self.logger.info(f"Unique URLs tracked: {len(self.downloaded_urls)}")
        self.logger.info(f"Unique hashes tracked: {len(self.downloaded_hashes)}")
        self.logger.info(f"Unique products tracked: {len(self.product_ids)}")

        return results


# Usage example
if __name__ == "__main__":
    scraper = ImprovedLamodaScraper()

    urls = [
        "https://www.lamoda.kz/c/477/clothes-muzhskaya-odezhda/?sitelink=topmenuM&l=2",
        "https://www.lamoda.kz/c/15/shoes-women/?sitelink=topmenuW&l=3",
        "https://www.lamoda.kz/c/557/accs-zhenskieaksessuary/?sitelink=topmenuW&l=4",
        "https://www.lamoda.kz/c/355/clothes-zhenskaya-odezhda/",
        "https://www.lamoda.kz/c/17/shoes-men/",
        "https://www.lamoda.kz/c/559/accs-muzhskieaksessuary/?sitelink=topmenuM&l=4"
    ]

    results = scraper.scrape_urls(
        urls, 
        max_items_per_url=10000,  # Reduced since we're getting unique images now
        max_pages_per_url=1000, 
        base_dir="images"
    )

    # Print results
    print("\n" + "="*50)
    print("SCRAPING SUMMARY")
    print("="*50)
    total_unique = 0
    for url, data in results.items():
        print(f"Category: {data['category_name']}")
        print(f"Downloaded: {data['downloaded_count']} unique images")
        total_unique += data['downloaded_count']
        print("-" * 30)
    
    print(f"TOTAL UNIQUE IMAGES: {total_unique}")
    print(f"DUPLICATE DETECTION PREVENTED: {scraper.stats['duplicates_skipped']} duplicates")
    print(f"UNDERWEAR PRODUCTS SKIPPED: {scraper.stats['underwear_skipped']} items")

2025-08-25 11:08:31,438 - INFO - Starting URL 1/6: clothes_muzhskaya_odezhda
2025-08-25 11:08:31,439 - INFO - Starting scraping: https://www.lamoda.kz/c/477/clothes-muzhskaya-odezhda/?sitelink=topmenuM&l=2 (max: 1000 items, 50 pages)
2025-08-25 11:08:33,108 - WARNING - Could not detect pagination: Malformed attribute selector at position 0
  line 1:
[[class*="load-more"]]
^
2025-08-25 11:08:33,123 - INFO - Scraping page 1: https://www.lamoda.kz/c/477/clothes-muzhskaya-odezhda/?sitelink=topmenuM&l=2&page=1
2025-08-25 11:08:37,605 - INFO - Found 420 products using selector: div[class*='x-product-card']
2025-08-25 11:08:37,661 - INFO - Downloaded: Поло_1.jpg (236x340)
2025-08-25 11:08:38,306 - INFO - URL already processed: https://a.lmcdn.ru/img236x341/M/P/MP002XM001T1_22472477_1_v1_2x.jpg
2025-08-25 11:08:39,755 - INFO - Downloaded: Олимпийка_2.jpg (236x340)
2025-08-25 11:08:40,440 - INFO - URL already processed: https://a.lmcdn.ru/img236x341/M/P/MP002XM0BC8L_24246688_1_v1_2x.jpg
2025-08

KeyboardInterrupt: 